# Multiclass Logistic Regression

Logistic regression extends naturally from the binary case to the case of $K > 2$ classes. The basic idea is the same: we build **linear scores** from the input, turn those scores into probabilities, and then fit the parameters by minimizing **cross-entropy loss**.

Throughout, let the response take values in
$$
y \in \{1,2,\dots,K\},
$$
and let the input be $x \in \mathbb{R}^D$.

Instead of using a single score function, we now use one score for each class:
$$
s_k(x) = w_k^\top x, \qquad k=1,\dots,K.
$$
Collecting these together gives a score vector
$$
s(x) =
\begin{pmatrix}
w_1^\top x \\
\vdots \\
w_K^\top x
\end{pmatrix}
\in \mathbb{R}^K.
$$

The interpretation is simple: larger score for class $k$ should mean class $k$ is more plausible for that input.


The prediction rule is
$$
\hat y(x) = \arg\max_{1 \le k \le K} s_k(x)  
= \arg\max_{1 \le k \le K} w_k^\top x.
$$

## One-Hot Encoding and Loss

It is convenient to encode the class label as a **one-hot vector**
$$
t = (t_1,\dots,t_K)^\top \in \{0,1\}^K,
$$
where exactly one component equals 1. If the observation belongs to class $c$ so that $y=c$ then $t_c = 1$ and all other coordinates are 0. Note that $y$ and $t$ are in one-to-one relation, so we'll switch between them with impunity.

To define a loss function we first convert the scores into probabilities, we use the **softmax** map:
$$
p_k(x)=\frac{e^{w_k^\top x}}{\sum_{j=1}^K e^{w_j^\top x}},
\qquad k=1,\dots,K.
$$

A few basic facts aout what softmax does:

- each $p_k(x)$ is positive
- the probabilities sum to 1
- the class with the largest probability is the same as the class with the largest score (monotonicity)

This is the multiclass analogue of the sigmoid link in binary logistic regression.


Given predicted probabilities $p(x) = (p_1(x),\dots,p_K(x))^\top$, the **multiclass cross-entropy loss** is defined as
$$
\ell(t, p(x)) = 
-\sum_{k=1}^K t_k \log p_k(x).
$$

Because only one component of $t$ is nonzero, this just selects the log-probability assigned to the correct class. If the true class is $y=c$, then
$$
\ell(y, p(x)) = \ell(c, p(x)) = -\log p_c(x) = -\log p_y(x).
$$

So the loss is small when the model assigns large probability to the correct class, and large when it does not.


Substituting the softmax formula gives a clean expression in terms of the linear scores. If the true class is $c$, then
$$
\ell(y, s(x)) =
-\log \left(
\frac{e^{w_c^\top x}}{\sum_{j=1}^K e^{w_j^\top x}}
\right) =
\log \left( \sum_{j=1}^K e^{w_j^\top x} \right) - w_c^\top x.
$$

More generally, using one-hot notation,
$$
\ell(y, s(x)) =
\log \left( \sum_{j=1}^K e^{w_j^\top x} \right) -
\sum_{k=1}^K y_k w_k^\top x.
$$

This is sometimes also called the **multinomial logistic loss** or **softmax loss**, but more commonly the **categorical cross entropy** loss. 


## ERM Formulation

Now suppose we observe data
$$
(x_1,y_1),\dots,(x_N,y_N),
$$
with each $y_n$ encoded as a one-hot vector in $t_n \in\mathbb{R}^K$.

Let
$$
W =
\begin{pmatrix}
\vert &        & \vert \\
w_1   & \cdots & w_K   \\
\vert &        & \vert
\end{pmatrix}
\in \mathbb{R}^{D \times K}
$$
be the parameter matrix whose $k$-th column is $w_k \in \mathbb{R}^D$.

For a single input $x_n$, the score vector is
$$
W^\top x_n \in \mathbb{R}^K,
$$
and the predicted probability vector is
$$
p(x_n;W) = \mathrm{softmax}(W^\top x_n).
$$

The empirical risk is then
$$
\hat R(W)=
\frac{1}{N}\sum_{n=1}^N
\left[
-\sum_{k=1}^K t_{nk} \log p_k(x_n;W)
\right].
$$

Our goal is to solve
$$
\hat W = \arg\min_W \hat R(W).
$$


## Matrix Form

Let

- $X \in \mathbb{R}^{N \times D}$ be the design matrix, with row $n$ equal to $x_n^\top$
- $T \in \mathbb{R}^{N \times K}$ be the matrix of one-hot labels
- $Z = XW \in \mathbb{R}^{N \times K}$ be the matrix of class scores
- $P = \mathrm{softmax}(Z)$, where softmax is applied rowwise (in a slight abuse of notation)

Then row $n$ of $P$ contains the predicted class probabilities for observation $n$.

In this notation, the empirical risk becomes
$$
\hat R(W)=-\frac{1}{N}
\sum_{n=1}^N \sum_{k=1}^K
T_{nk} \log P_{nk}.
$$

This is very similar to the binary logistic regression objective, except that the scalar sigmoid probabilities are replaced by row vectors of softmax probabilities.


## Identifiability Issues

As previously, there are some identifiability issues. If we add the same vector $a \in \mathbb{R}^D$ to every class weight vector, then the probabilities do not change:
$$
w_k \mapsto w_k + a \qquad \text{for all } k.
$$

Why? Because each score increases by the same amount $a^\top x$, and softmax is unchanged when the same constant is added to all coordinates.

So the parameterization is **redundant**: many different matrices $W$ define exactly the same classifier.

Common ways to handle this are:

- fix one class as a reference class, for example set $w_K = 0$ (classical manner)
- rely on numerical solvers that handle the redundancy internally (modern ML YOLO lifestyle)

Note that this issue affects the parameters and their optimization, not the predicted probabilities. Modern thinking: if we are happy enough with our optimizer results (they're not too unstable) then its probably not a big deal. The more classical thinking says that we want to interpret coefficients, so its better just to pin a "base class".

## Gradient Descent

As before, the objective is not quadratic in the parameters, so there is no normal-equation style formula for $\hat W$.

We therefore use an iterative optimization method such as:

- gradient descent
- Newton or quasi-Newton methods
- stochastic gradient methods

The good news is that the gradient still has a very clean form.



We first compute the gradient of the empirical risk with respect to the whole matrix $W$.

For softmax, a standard derivative identity is
$$
\frac{\partial P_{na}}{\partial Z_{nb}} =
P_{na}(\mathbf{1}\{a=b\} - P_{nb}).
$$

Using this together with the chain rule, the gradient of the empirical risk is
$$
\nabla_W \hat R(W) =
\frac{1}{N} X^\top (P - T).
$$

This is nicely similar to what we derived in the binary classification case. Isn't that nice!? It has almost exactly the same shape as in the binary case. The only difference is that now the residual is matrix-valued:
$$
P - T \in \mathbb{R}^{N \times K}.
$$

**Side comment.**  
We just wrote a derivative with respect to a matrix $W$, which can feel like a new step. In practice, this is mostly bookkeeping. The gradient with respect to $W \in \mathbb{R}^{D \times K}$ is just the collection of gradients with respect to each column $w_k$, arranged into a matrix of the same shape. What is slightly new is the notation: instead of writing $K$ separate vector derivatives, we package everything into one object. This lets us write updates and derivations more compactly, but does not change the underlying calculations. 

The rule: 
> When differentiating a scalar function, the gradient has the same shape as the argument. Matrix gradients are just a convenient way of organizing many partial derivatives.

So gradient descent takes the form
$$
W^{(t+1)}=
W^{(t)} - \eta \frac{1}{N} X^\top (P^{(t)} - T),
$$
where
$$
P^{(t)} = \mathrm{softmax}(X W^{(t)}).
$$


Each iteration is:

1. compute the score matrix $XW^{(t)}$
2. apply the rowwise softmax to obtain $P^{(t)}$
3. compute the gradient $\frac{1}{N}X^\top(P^{(t)} - Y)$
4. update $W$

This is a direct generalization of binary logistic regression. (Here, we are ignoring the parameterization issue.)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(scores):
    scores = np.asarray(scores)
    # numerical stability hack
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=-1, keepdims=True)

def cross_entropy_onehot(t, p):
    return -np.sum(t * np.log(p))

def softmax_loss(X, T, W):
    scores = X @ W
    P = softmax(scores)
    return -np.sum(T * np.log(P)) / X.shape[0]

def softmax_gradient(X, T, W):
    P = softmax(X @ W)
    return X.T @ (P - T) / X.shape[0]

def softmax_regression_gd(X, T, W0=None, eta=0.1, max_iter=5000, tol=1e-8):
    n_features = X.shape[1]
    n_classes = T.shape[1]
    W = np.zeros((n_features, n_classes)) if W0 is None else W0.copy()
    W_hist = [W]
    loss_history = [softmax_loss(X, T, W)]

    for _ in range(max_iter):
        grad = softmax_gradient(X, T, W)
        W = W - eta * grad
        W_hist.append(W)
        loss_history.append(softmax_loss(X, T, W))
        if np.linalg.norm(grad) < tol:
            break

    return W, np.array(loss_history), W_hist


In [ ]:
import matplotlib.pyplot as plt

def one_hot(y, K):
    T = np.zeros((len(y), K))
    T[np.arange(len(y)), y] = 1
    return T

In [ ]:
def simulate_softmax_data(n=300, d=2, K=3, W_true=None, seed=None):
    rng = np.random.default_rng(seed)

    # covariates
    X = rng.normal(size=(n, d))

    # add intercept
    X = np.column_stack([np.ones(n), X])

    # true parameter matrix
    W_true = rng.normal(loc=1,scale=1.5,size=(d+1,K))

    P = softmax(X @ W_true)

    y = np.array([rng.choice(K, p=P[i]) for i in range(n)])
    T = one_hot(y, K)

    return X, y, T, W_true, P

In [ ]:
D = 2
K = 3
X, y, T, W_true, P_true = simulate_softmax_data(n=1000, d=D, K=K, seed=None)

In [ ]:
T.sum(axis=0)

In [ ]:
plt.figure(figsize=(5,5))

plt.scatter(X[y==0,1],X[y==0,2],label="y = 0")
plt.scatter(X[y==1,1],X[y==1,2],label="y = 1")
plt.scatter(X[y==2,1],X[y==2,2],label="y = 2")
plt.scatter(X[y==3,1],X[y==3,2],label="y = 3")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("2D dataset")
plt.legend()
plt.grid(True)
plt.axis("equal")

plt.show()

In [ ]:
W_hat, loss_hist, W_hist = softmax_regression_gd(
    X, T,
    eta=0.5,
    max_iter=10000,
    tol=1e-6
)

In [ ]:
def predict_proba(X, W):
    return softmax(X @ W)

def predict_class(X, W):
    return np.argmax(predict_proba(X, W), axis=1)

def classification_error(y_true, y_pred):
    return np.mean(y_true != y_pred)

In [ ]:
P_hat = predict_proba(X, W_hat)
P_hat[:5]

In [ ]:
y_hat = predict_class(X, W_hat)
y_hat[:5]

In [ ]:
print("Training loss at truth:   ", softmax_loss(X, T, W_true))
print("Training loss at estimate:", softmax_loss(X, T, W_hat))
print("Training classification error at truth:   ", classification_error(y, np.argmax(P_true, axis=1)))
print("Training classification error at estimate:", classification_error(y, y_hat))

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(loss_hist)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy loss")
plt.title("Gradient descent for multiclass logistic regression")
plt.show()

We can see how these converge over the iterations, though its a bit more difficult than in the binary case. 

In [ ]:
W_hist = np.array(W_hist)

In [ ]:
W_hist.shape

In [ ]:
fig, axes = plt.subplots(1, K, figsize=(D*K, D), sharey=True)

for k in range(K):
    ax = axes[k] if K > 1 else axes
    for d in range(D):
        ax.plot(W_hist[:, d, k], label=f"d={d}")
        ax.axhline(W_true[d, k], linestyle='--', color='black', alpha=0.7)
    ax.set_title(f"class k={k}")
    ax.set_xlabel("Iteration")
    if k == 0:
        ax.set_ylabel("Value")
    ax.legend()

plt.show()

We really need to contend with some non-identifiabiltiy issues if we want to compare against the dashed "true" lines.

If we use the *last* class as the identifiability constraint, we subtract that column from all the others in $W$. 

In [ ]:
k_ref = K-1

fig, axes = plt.subplots(1, K, figsize=(D*K, D), sharey=True)

for k in range(K):
    ax = axes[k] if K > 1 else axes
    for d in range(D):
        ax.plot(W_hist[:, d, k] - W_hist[:, d, k_ref], label=f"d={d}")
        ax.axhline(W_true[d, k] - W_true[d, k_ref], linestyle='--', color='black', alpha=0.7)
    ax.set_title(f"class k={k}")
    ax.set_xlabel("Iteration")
    if k == 0:
        ax.set_ylabel("Value")
    ax.legend()

plt.show()

Notice how the $K=3$ reference class now has zeroed out weights.

## Decision Rule and Decision Boundaries

The prediction rule is
$$
\hat y(x) = \hat{f}(x) = \arg\max_k w_k^\top x.
$$

This means the boundary between classes $a$ and $b$ is determined by where their scores tie:
$$
w_a^\top x = w_b^\top x.
$$

Equivalently,
$$
(w_a - w_b)^\top x = 0.
$$

So with raw linear features, the boundary between any pair of classes is still linear. In multiclass logistic regression, the input space gets partitioned into several regions, one region per class, separated by pairwise linear boundaries. We'll end up calling this a **linear classifier.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_multiclass_decision_regions(X, y, W, grid_points=300, pad=0.6, title="Decision regions"):
    x1 = X[:, 1]
    x2 = X[:, 2]
    K = W.shape[1]

    x1_min, x1_max = x1.min() - pad, x1.max() + pad
    x2_min, x2_max = x2.min() - pad, x2.max() + pad

    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, grid_points),
        np.linspace(x2_min, x2_max, grid_points)
    )

    X_grid = np.column_stack([
        np.ones(xx1.size),
        xx1.ravel(),
        xx2.ravel()
    ])

    P_grid = softmax(X_grid @ W)
    y_grid = np.argmax(P_grid, axis=1).reshape(xx1.shape)

    # choose a consistent colormap
    cmap = plt.get_cmap("tab10")
    colors = [cmap(k) for k in range(K)]

    plt.figure(figsize=(7, 6))

    # background regions
    plt.contourf(
        xx1, xx2, y_grid,
        levels=np.arange(K + 1) - 0.5,
        colors=colors,
        alpha=0.25
    )

    # boundary lines
    plt.contour(
        xx1, xx2, y_grid,
        levels=np.arange(K - 1) + 0.5,
        linewidths=1,
        colors='k'
    )

    # data points with matching colors
    for k in range(K):
        mask = (y == k)
        plt.scatter(
            x1[mask],
            x2[mask],
            color=colors[k],
            edgecolor='k',
            s=40,
            label=f"class {k}"
        )

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
plot_multiclass_decision_regions(X, y, W_hat, title="Estimated multiclass decision regions")

# Code Examples

Below we fit a multiclass logistic regression model in two ways:

1. using our own simple gradient descent implementation
2. using `sklearn`'s multinomial logistic regression solver

We will use the three-species penguins dataset with two features so that the class regions can be visualized directly.


In [ ]:
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import DecisionBoundaryDisplay

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (6, 4)
np.set_printoptions(suppress=True, precision=4)


## Penguins

Let's work with the [penguins](https://allisonhorst.github.io/palmerpenguins/) dataset. The dataset is loaded directly from the CSV gist in the code below.


In [ ]:
url = "https://gist.githubusercontent.com/slopp/ce3b90b9168f2f921784de84fa445651/raw/penguins.csv"
penguins = pd.read_csv(url)
penguins = penguins.dropna().copy().sample(frac=1, random_state=1234)

cols = ["flipper_length_mm", "bill_length_mm", "species"]
penguins = penguins[cols]
penguins.head()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    data=penguins,
    x="flipper_length_mm",
    y="bill_length_mm",
    hue="species",
    ax=ax,
)
ax.set_title("Penguins: three species")
plt.show()


Let's try to implement it ourselves:

We use two predictors:

- flipper length
- bill depth

and keep all three species, so this is a genuine multiclass problem.


In [ ]:
X_raw = penguins[["flipper_length_mm", "bill_length_mm"]].to_numpy()
y = penguins["species"].to_numpy()

print(X_raw[:5])
print(y[:5])

In [ ]:
classes = np.unique(y)
class_to_int = {c: i for i, c in enumerate(classes)}
y_int = np.array([class_to_int[c] for c in y])
print(y_int[:5])

In [ ]:
T = np.eye(len(classes))[y_int]
T[:5]

In [ ]:
X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0)
X_stdzd = (X_raw - X_mean) / X_std
X_design = np.column_stack([np.ones(len(X_stdzd)), X_stdzd])
X_design[:5]

In [ ]:
print("Classes:", classes)
print("Design matrix shape:", X_design.shape)
print("One-hot label matrix shape:", Y.shape)

In [ ]:
W_hat, loss_history, W_hist = softmax_regression_gd(
    X_design, T, eta=0.5, max_iter=15000, tol=1e-7
)

W_hat

Note, again, this is not *exactly* unique.

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy loss")
plt.title("Gradient descent for multiclass logistic regression")
plt.show()

Given the fitted parameter matrix, we compute the class probabilities for each row and then predict the most likely class.

In [ ]:
P_hat = softmax(X_design @ W_hat)
print(P_hat[:5])

In [ ]:
pred_int = np.argmax(P_hat, axis=1)
pred = classes[pred_int]
print(pred[:5])

**Confusion matrix**

For multiclass classification, the confusion matrix is now a $K \times K$ table.

- rows are the true classes
- columns are the predicted classes

The diagonal entries count correct predictions, while off-diagonal entries show which classes the model confuses.


In [ ]:
cm = confusion_matrix(y, pred, labels=classes)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(ax=ax, colorbar=False)
ax.grid(False)
plt.show()

Precision, recall, and F1 score all generalize to the multiclass setting by computing them class-by-class.

In [ ]:
print(classification_report(y, pred, target_names=classes))

Now we fit the standard multinomial logistic regression model from `sklearn`.

In [ ]:
sk_mod = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=np.inf)
)

sk_mod.fit(X_raw, y)

In [ ]:
print("Intercepts:")
print(sk_mod.named_steps["logisticregression"].intercept_)

print("\nCoefficients:")
print(sk_mod.named_steps["logisticregression"].coef_)


Because of parameter redundancy, these coefficients do not need to match the ones from our own implementation exactly. In any case, compare:

In [ ]:
W_hat

made identifiable:

In [ ]:
W_hat - W_hat[:, [2]]

In [ ]:
W_skl = np.vstack([
    sk_mod.named_steps["logisticregression"].intercept_[None, :],
    sk_mod.named_steps["logisticregression"].coef_.T
])
W_skl

made identifiable:

In [ ]:
(W_skl - W_skl[:, [2]])

Surprisingly good match!

What should match are the fitted probabilities, predictions, and decision regions up to numerical tolerance.

In [ ]:
sk_probs = sk_mod.predict_proba(X_raw)
sk_probs[:5]

In [ ]:
P_hat[:5]

In [ ]:
sk_pred = sk_mod.predict(X_raw)
sk_pred[:10]

In [ ]:
pred[:10]

In [ ]:
print("Training accuracy from our gradient descent fit:", np.mean(pred == y))
print("Training accuracy from sklearn:", np.mean(sk_pred == y))

## Plot the decision regions

The multiclass model partitions the plane into one region per species. Each boundary between two neighboring regions is linear in these features.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))

DecisionBoundaryDisplay.from_estimator(
    sk_mod,
    X_raw,
    response_method="predict",
    alpha=0.25,
    ax=ax,
)

sns.scatterplot(
    data=penguins,
    x="flipper_length_mm",
    y="bill_length_mm",
    hue="species",
    ax=ax,
    edgecolor="black",
    linewidth=0.5,
)

ax.set_title("Multiclass logistic regression decision regions")
plt.show()


## Feature Engineering

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
quad_mod = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=3, include_bias=False),
    LogisticRegression(C=np.inf,max_iter=10000)
)
quad_mod.fit(X_raw, y)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))

DecisionBoundaryDisplay.from_estimator(
    quad_mod,
    X_raw,
    response_method="predict",
    alpha=0.25,
    ax=ax,
)

sns.scatterplot(
    data=penguins,
    x="flipper_length_mm",
    y="bill_length_mm",
    hue="species",
    ax=ax,
    edgecolor="black",
    linewidth=0.5,
)

ax.set_title("Multiclass logistic regression decision regions")
plt.show()


# MNIST

Let's look as classifying hand-written images using the [MNIST](https://en.wikipedia.org/wiki/MNIST_database) dataset.

In [ ]:
from sklearn.datasets import fetch_openml

In [ ]:
X, y = fetch_openml("mnist_784", version=1, as_frame=False, return_X_y=True)
y = y.astype(int)
X = X / 255.0 #rescale to 0-1

# subset
n = 20000
X = X[:n]
y = y[:n]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

In [ ]:
def plot_mnist_images(X, y=None, n=10, nrows=2, figsize=(8, 4), random=True):
    N = X.shape[0]
    ncols = int(np.ceil(n / nrows))

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)

    axes = np.array(axes).reshape(-1)

    for i, ax in enumerate(axes):
        if i >= n:
            ax.axis("off")
            continue

        img = X[i].reshape(28, 28)
        ax.imshow(img, cmap="gray")
        ax.axis("off")

        if y is not None:
            ax.set_title(str(y[i]))

    plt.tight_layout()
    plt.show()

In [ ]:
plot_mnist_images(X_train)

In [ ]:
mod = LogisticRegression(C=np.inf,
                        solver="lbfgs",
                        fit_intercept=True,
                        max_iter=10000)
mod.fit(X_train, y_train)

In [ ]:
y_pred = mod.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.title("Confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for k, ax in enumerate(axes.ravel()):
    ax.imshow(mod.coef_[k].reshape(28, 28), cmap="gray")
    ax.set_title(f"class {k}")
    ax.axis("off")

plt.suptitle("Learned weight vectors (one per class)")
plt.show()

# One-vs-rest 

There are two common ways to extend binary logistic regression to multiple classes.

**One-vs-rest**

Fit $K$ separate binary classifiers:
- class 1 versus not class 1: $s_1$
- class 2 versus not class 2: $s_2$
- and so on

This is simple, but the resulting score functions need not fit together as a single coherent multiclass model. We can still do something like: 

$$
\hat{y} = \hat{f}(x) = \arg\max_k s_k(x)
$$

and it tends to work reasonably well.

**Multinomial logistic regression** What we have done here. 

# Linear classifiers.
In general, we call any classifier a **linear classifier** if it can be written such that for class $k$, we compute
$$
s_k(x) = h(w_k^\top x),
$$
where $w_k \in \mathbb{R}^D$ and $h:\mathbb{R}\to\mathbb{R}$ is a monotone function.

Because $h$ is monotone, it preserves order. This means the predicted class
$$
\hat{f}(x) = \arg\max_k s_k(x)
$$
is unchanged if we replace $s_k(x)$ by $w_k^\top x$. In other words, the decision rule depends only on the linear functions $w_k^\top x$.

As a result, linear classifiers partition the input space using linear **decision boundaries**. To see this, consider two classes $k$ and $j$. The boundary between them is given by the set of points where their scores are equal:
$$
s_k(x) = s_j(x).
$$
Since $h$ is monotone, this is equivalent to
$$
w_k^\top x = w_j^\top x,
$$
or
$$
(w_k - w_j)^\top x = 0.
$$
This is a linear equation in $x$, so the boundary is a hyperplane.

# Problem Ideas

- derive the softmax gradient carefully and show that it reduces to $X^\top(P-Y)$
- show that adding the same vector to every class weight leaves the probabilities unchanged
- derive the pairwise class boundary equation $(w_a - w_b)^\top x = 0$
- compare one-vs-rest and multinomial logistic regression on the same dataset
- add polynomial features and visualize how the decision regions change
- derive the Hessian block formula and explain why Newton's method becomes more expensive
